In [ ]:
# 1. 必要なライブラリのインストール
!pip install umap-learn scipy

import os
import shutil
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import pairwise_distances
import umap
from scipy.stats import spearmanr
from google.colab import files
import warnings
warnings.filterwarnings('ignore')

# 2. 実験設定とディレクトリ作成
OUT_DIR = "H2_experiment_results"
os.makedirs(OUT_DIR, exist_ok=True)

# 試行回数（総生成要素数 N_samples * D_features）の6分割スケール
trial_scales = [1_000, 10_000, 100_000, 1_000_000, 5_000_000, 10_000_000]

# N_SAMPLESを固定し、次元数(D)を拡張する（ペア数は 400*399/2 = 79,800ペア）
N_SAMPLES = 400
results = []

print("=== H2計算実験（大域的距離の無意味化）を開始します ===")

# 3. メインループ（6分割スケール）
for i, total_trials in enumerate(trial_scales):
    step = i + 1
    D_features = max(2, total_trials // N_SAMPLES)
    print(f"\n[Step {step}/6] 試行回数: {total_trials:,} (N={N_SAMPLES}, D={D_features})")

    # --- A. 初期値依存性を排除したベースライン生成 ---
    np.random.seed(42 + step)
    X_base = np.random.uniform(0, 0.2, size=(N_SAMPLES, D_features))

    # --- B. AHP的偏り・過分散の注入 ---
    X_skewed = X_base.copy()
    hub_dims = max(1, int(D_features * 0.05))
    skew_multiplier = 10.0 + (step * 5.0)
    X_skewed[:, :hub_dims] *= skew_multiplier

    # --- C. 高次元空間の距離行列（Cosine Distance） ---
    dist_high_base = pairwise_distances(X_base + 1e-10, metric='cosine')
    dist_high_skewed = pairwise_distances(X_skewed + 1e-10, metric='cosine')

    # --- D. UMAPによる次元圧縮 ---
    reducer = umap.UMAP(n_components=2, metric='precomputed', random_state=42)
    umap_base = reducer.fit_transform(dist_high_base)
    umap_skewed = reducer.fit_transform(dist_high_skewed)

    # --- E. 低次元（UMAP）空間の距離行列（Euclidean Distance） ---
    dist_low_base = pairwise_distances(umap_base, metric='euclidean')
    dist_low_skewed = pairwise_distances(umap_skewed, metric='euclidean')

    # --- F. キャリブレーション（順位相関の計算） ---
    # 行列の上三角成分（対角成分を除く全ペアの距離）を1次元配列として抽出
    iu = np.triu_indices(N_SAMPLES, k=1)

    flat_high_base = dist_high_base[iu]
    flat_low_base = dist_low_base[iu]
    flat_high_skewed = dist_high_skewed[iu]
    flat_low_skewed = dist_low_skewed[iu]

    # スピアマンの順位相関を計算（大域的距離の保持度合い）
    rho_base, p_base = spearmanr(flat_high_base, flat_low_base)
    rho_skewed, p_skewed = spearmanr(flat_high_skewed, flat_low_skewed)

    # 結果の記録
    results.append({
        'Step': step,
        'Total_Trials': total_trials,
        'Dimensions': D_features,
        'Spearman_Rho_Base': rho_base,
        'Spearman_Rho_Skewed': rho_skewed
    })

    # --- G. グラフ出力（都度保存） ---
    fig, axes = plt.subplots(2, 2, figsize=(12, 10))
    fig.suptitle(f'Step {step}: Trials={total_trials:,} (D={D_features})', fontsize=16)

    # 上段: UMAP散布図
    axes[0,0].scatter(umap_base[:,0], umap_base[:,1], s=10, alpha=0.6)
    axes[0,0].set_title('UMAP: Baseline (Uniform)')
    axes[0,1].scatter(umap_skewed[:,0], umap_skewed[:,1], s=10, alpha=0.6, color='red')
    axes[0,1].set_title('UMAP: Skewed (Topology Collapsed)')

    # 下段: 高次元距離 vs 低次元距離のHexbinプロット（相関の視覚化）
    hb1 = axes[1,0].hexbin(flat_high_base, flat_low_base, gridsize=30, cmap='Blues', bins='log')
    axes[1,0].set_title(f'Distance Correlation (Baseline)\nSpearman $\\rho$: {rho_base:.3f}')
    axes[1,0].set_xlabel('High-D Distance (Cosine)')
    axes[1,0].set_ylabel('Low-D Distance (Euclidean)')

    hb2 = axes[1,1].hexbin(flat_high_skewed, flat_low_skewed, gridsize=30, cmap='Reds', bins='log')
    axes[1,1].set_title(f'Distance Correlation (Skewed)\nSpearman $\\rho$: {rho_skewed:.3f}')
    axes[1,1].set_xlabel('High-D Distance (Cosine)')
    axes[1,1].set_ylabel('Low-D Distance (Euclidean)')

    plt.tight_layout()
    plt.savefig(f"{OUT_DIR}/trial_step_{step}.png")
    plt.close()

# 4. 結果の集計とまとめグラフの作成
df_results = pd.DataFrame(results)
df_results.to_csv(f"{OUT_DIR}/summary_results.csv", index=False)

# スピアマン順位相関の推移グラフ（キャリブレーション証明）
plt.figure(figsize=(8, 6))
plt.plot(df_results['Total_Trials'], df_results['Spearman_Rho_Base'], marker='o', label='Baseline (Uniform)')
plt.plot(df_results['Total_Trials'], df_results['Spearman_Rho_Skewed'], marker='s', color='red', label='Skewed (Hubness Injected)')
plt.xscale('log')
plt.ylim(0, 1.0)
plt.xlabel('Total Trials (N_samples * D_features, log scale)')
plt.ylabel("Spearman's Rank Correlation ($\\rho$)")
plt.title('Degradation of Global Distance Meaningfulness')
plt.grid(True, which="both", ls="--")
plt.legend()
plt.tight_layout()
plt.savefig(f"{OUT_DIR}/summary_correlation_plot.png")
plt.close()

# 5. ZIP圧縮とダウンロード
shutil.make_archive(OUT_DIR, 'zip', OUT_DIR)
print(f"\n実験完了。全データを {OUT_DIR}.zip に圧縮しました。ダウンロードを開始します...")
files.download(f"{OUT_DIR}.zip")

=== H2計算実験（大域的距離の無意味化）を開始します ===

[Step 1/6] 試行回数: 1,000 (N=400, D=2)

[Step 2/6] 試行回数: 10,000 (N=400, D=25)

[Step 3/6] 試行回数: 100,000 (N=400, D=250)

[Step 4/6] 試行回数: 1,000,000 (N=400, D=2500)

[Step 5/6] 試行回数: 5,000,000 (N=400, D=12500)

[Step 6/6] 試行回数: 10,000,000 (N=400, D=25000)

実験完了。全データを H2_experiment_results.zip に圧縮しました。ダウンロードを開始します...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>